# 1. Inspect compact data
Run top to bottom in a fresh kernel. Configure paths and the signal mass in
`settings.local.json` (copy `settings.example.json` first). No ROOT files are needed.

This notebook validates complete exports before plotting. It does not prepare
splits, train a model, or alter compact data. Numerical reports and figures go
under the configured workspace; notebook output should be cleared before commit.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from hepml.adapters.research import (
    read_settings, inspect_inputs, feature_preview, save_provenance,
    benchmark_directory, prepare_benchmark, train_benchmark,
    stage_status, check_splits, compare_runs,
)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/hepml").is_dir())
SETTINGS = Path(os.environ.get("HEPML_RESEARCH_CONFIG", REPO / "notebooks/settings.local.json"))
config = read_settings(SETTINGS)
display(pd.Series(config, name="Effective settings"))
print(f"Luminosity: {config['analysis']['lumi']:g} pb^-1 ({config['analysis']['lumi'] / 1000:g} fb^-1)")


## Validate inputs and review normalization
Checks all shard hashes, metadata, source identities and the extraction recipe.
Each row is one sample. Signal hypotheses are alternatives: do **not** sum their yields.
`n_root` counts entries before compact selection; `normalization_count` may differ
if production bookkeeping explicitly overrides it. Yields use uniform weights
`luminosity * xs_pb / normalization_count`, not raw generator weights.

In [ ]:
inventory, records = inspect_inputs(config)
display(inventory)
selected_inventory = inventory[(inventory.kind == "background") | (inventory.mass == config["mass"])]
display(selected_inventory.groupby("kind")[["n_selected", "selected_yield"]].sum())

## Inspect derived features
Preview the first configured number of accepted events per sample, in shard order.
This is deterministic but **not a random representative sample**. Full event counts
and yields are shown above. Preview histograms and correlations are exploratory;
use training-only data when deciding transformations or features for a locked benchmark.

In [ ]:
preview = feature_preview(config, records)
features = config["features"]
display(preview.groupby(["sample", "label"]).size().rename("preview_events").to_frame())
display(preview[features].describe().T)
print("Finite feature values, preserved event order and unique preview event IDs: passed")

## Feature shapes
Each class histogram has unit area and uses physical weights within the preview.
The prefix cap can distort the background mixture; these plots are not yield estimates.

In [ ]:
import numpy as np
columns = 3
fig, axes = plt.subplots(int(np.ceil(len(features) / columns)), columns, figsize=(13, 3 * int(np.ceil(len(features) / columns))))
for ax, feature in zip(np.asarray(axes).ravel(), features):
    edges = np.histogram_bin_edges(preview[feature], bins=35)
    for label, name in [(0, "Background"), (1, "Signal")]:
        group = preview[preview.label == label]
        ax.hist(group[feature], bins=edges, weights=group.sample_weight, density=True, histtype="step", label=name)
    ax.set(xlabel=feature, ylabel="Unit-area preview density")
    ax.legend(fontsize=8)
for ax in np.asarray(axes).ravel()[len(features):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

## Correlations
Unweighted Pearson correlations, separately for signal and background prefixes.
These are descriptive checks, not a feature-selection score.

In [ ]:
corr_fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for ax, label, name in zip(axes, [0, 1], ["Background preview", "Signal preview"]):
    corr = preview.loc[preview.label == label, features].corr()
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set(title=name, xticks=range(len(features)), yticks=range(len(features)))
    ax.set_xticklabels(features, rotation=90)
    ax.set_yticklabels(features)
corr_fig.colorbar(im, ax=axes, label="Pearson correlation")
plt.show()

## Save this inspection
Creates a new timestamped report directory. Preserves full manifests, source
snapshots (including uncommitted edits), effective settings and package versions.
Keep observations alongside these reports. No existing report is overwritten.

In [ ]:
from datetime import datetime, timezone
report_dir = Path(config["workspace"]) / "reports" / ("inspection-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
report_dir.mkdir(parents=True, exist_ok=False)
save_provenance(report_dir, config, records)
inventory.to_csv(report_dir / "sample_inventory.csv", index=False)
preview[features].describe().T.to_csv(report_dir / "feature_summary.csv")
fig.savefig(report_dir / "feature_shapes.png", dpi=150, bbox_inches="tight")
corr_fig.savefig(report_dir / "correlations.png", dpi=150, bbox_inches="tight")
print(report_dir)